# Background and disturbance checks — Coimbra pilot

**Purpose.** The two new checks for the next meeting
(`docs/meeting_notes.md`): presences vs. a candidate target-group
background, and the effect of the proposed burned-area filter. Year
distribution and observer concentration are already in
`../02_citizen_science/acacia_coimbra_pilot_exploration.ipynb`.

**Scope.** Pilot area only (iNaturalist place 8422, Coimbra district).
The window (2021–2025, reference year 2023), the target-group
background and the disturbance rule are **proposals** from
`docs/research_questions.md`, not decisions. Downloads are cached in
`data/raw/` and documented in `data/README.md`; figures go to
`results/figures/meeting_2026-10/` (not versioned).

In [ ]:
import json
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import requests
from shapely.geometry import shape

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW_CS = ROOT / "data" / "raw" / "citizen_science"
RAW_FIRE = ROOT / "data" / "external" / "auxiliary" / "icnf_burned_areas"
FIG = ROOT / "results" / "figures" / "meeting_2026-10"
TAB = ROOT / "results" / "tables" / "meeting_2026-10"
for d in (RAW_FIRE, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

PLACE_ID = 8422          # iNaturalist "Coimbra, PT" (district)
ACACIA = 47452           # genus Acacia
PLANTAE = 47126          # kingdom Plantae
WINDOW = (2021, 2025)    # proposed observation window
REF_YEAR = 2023          # proposed reference year
METRIC_CRS = 3763        # PT-TM06/ETRS89

## 1. Download: other-plant observations (candidate target-group background)

In [ ]:
BG_FILE = RAW_CS / "inaturalist_plantae_no_acacia_coimbra_2021_2025.csv"
PARTIAL = BG_FILE.with_suffix(".partial.csv")

def download_background():
    """Resumable: appends each page to PARTIAL and restarts from the last id saved."""
    url = "https://api.inaturalist.org/v1/observations"
    last_id = int(pd.read_csv(PARTIAL, usecols=["id"])["id"].max()) if PARTIAL.exists() else 0
    params = dict(place_id=PLACE_ID, taxon_id=PLANTAE, without_taxon_id=ACACIA,
                  d1=f"{WINDOW[0]}-01-01", d2=f"{WINDOW[1]}-12-31",
                  order_by="id", order="asc", per_page=200, id_above=last_id)
    while True:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        res = r.json()["results"]
        if not res:
            break
        rows = []
        for o in res:
            if not o.get("geojson"):
                continue
            lon, lat = o["geojson"]["coordinates"]
            t = o.get("taxon") or {}
            rows.append(dict(id=o["id"], observed_on=o.get("observed_on"),
                             taxon_id=t.get("id"), taxon_name=t.get("name"), taxon_rank=t.get("rank"),
                             latitude=lat, longitude=lon,
                             positional_accuracy=o.get("positional_accuracy"),
                             quality_grade=o.get("quality_grade"),
                             user_login=(o.get("user") or {}).get("login"),
                             license_code=o.get("license_code")))
        pd.DataFrame(rows).to_csv(PARTIAL, mode="a", header=not PARTIAL.exists(), index=False)
        params["id_above"] = res[-1]["id"]
        time.sleep(1)  # iNaturalist asks for <= 60 requests/min
    PARTIAL.rename(BG_FILE)

if not BG_FILE.exists():
    download_background()
bg_raw = pd.read_csv(BG_FILE)
len(bg_raw)

Obscured coordinates (user or taxon geoprivacy) are randomised within
a ~0.2° cell — unusable at 10 m. Their ids are fetched separately and
excluded below (the download above did not keep that flag).

In [ ]:
OBSCURED_FILE = RAW_CS / "inaturalist_plantae_no_acacia_coimbra_2021_2025_obscured_ids.csv"

def download_obscured_ids():
    url = "https://api.inaturalist.org/v1/observations"
    ids = set()
    for privacy in [dict(geoprivacy="obscured,private"), dict(taxon_geoprivacy="obscured,private")]:
        params = dict(place_id=PLACE_ID, taxon_id=PLANTAE, without_taxon_id=ACACIA,
                      d1=f"{WINDOW[0]}-01-01", d2=f"{WINDOW[1]}-12-31", order_by="id", order="asc",
                      per_page=200, id_above=0, only_id="true", **privacy)
        while True:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            res = r.json()["results"]
            if not res:
                break
            ids.update(o["id"] for o in res)
            params["id_above"] = res[-1]["id"]
            time.sleep(1)
    pd.DataFrame({"id": sorted(ids)}).to_csv(OBSCURED_FILE, index=False)

if not OBSCURED_FILE.exists():
    download_obscured_ids()
obscured_ids = set(pd.read_csv(OBSCURED_FILE)["id"])
len(obscured_ids)

## 2. Download: ICNF burned areas 2021–2025

In [ ]:
def download_fires(year):
    f = RAW_FIRE / f"ardida_{year}.geojson"
    if not f.exists():
        url = "https://si.icnf.pt/wfs/areas_ardidas"
        params = dict(service="WFS", version="2.0.0", request="GetFeature",
                      typeNames=f"BDG:ardida_{year}", outputFormat="application/json")
        r = requests.get(url, params=params, timeout=300)
        r.raise_for_status()
        f.write_bytes(r.content)
    return gpd.read_file(f).set_crs(METRIC_CRS, allow_override=True)

fires = pd.concat([download_fires(y) for y in range(WINDOW[0], WINDOW[1] + 1)], ignore_index=True)
fires["fire_date"] = pd.to_datetime(fires["DH_Inicio"], utc=True).dt.tz_localize(None)
fires.groupby("Ano").size()

## 3. Load and prepare

In [ ]:
place = requests.get(f"https://api.inaturalist.org/v1/places/{PLACE_ID}", timeout=60).json()["results"][0]
district = gpd.GeoDataFrame(geometry=[shape(place["geometry_geojson"])], crs=4326).to_crs(METRIC_CRS)

def to_gdf(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["observed_on"], errors="coerce")
    df["year"] = df["date"].dt.year
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs=4326).to_crs(METRIC_CRS)

acacia_raw = pd.read_csv(RAW_CS / "inaturalist_acacia_coimbra_pilot.csv")
acacia = to_gdf(acacia_raw[~acacia_raw["coordinates_obscured"]])
bg = to_gdf(bg_raw[~bg_raw["id"].isin(obscured_ids)])
print(f"Dropped obscured: Acacia {len(acacia_raw) - len(acacia)}, background {len(bg_raw) - len(bg)}")

# The dominant observer is anonymised as "Observer A"
dominant = acacia["user_login"].value_counts().index[0]
for df in (acacia, bg):
    df["observer"] = (df["user_login"] == dominant).map({True: "Observer A", False: "Others"})

in_window = lambda df: df[df["year"].between(*WINDOW)]
acacia_w, bg_w = in_window(acacia), in_window(bg)
print(f"Acacia: {len(acacia)} total, {len(acacia_w)} in window; background: {len(bg_w)} in window")

def base(ax, title):
    district.boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_title(title); ax.set_axis_off()

## 4. Presences vs. candidate background (window 2021–2025)

In [ ]:
groups = [("All observers", acacia_w, bg_w),
          ("Observer A", acacia_w[acacia_w.observer == "Observer A"], bg_w[bg_w.observer == "Observer A"]),
          ("Others", acacia_w[acacia_w.observer == "Others"], bg_w[bg_w.observer == "Others"])]
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharex=True, sharey=True)
for ax, (label, p, b) in zip(axes, groups):
    b.plot(ax=ax, markersize=0.5, color="#7f7f7f", alpha=0.3, label=f"other plants (n={len(b)})")
    p.plot(ax=ax, markersize=2, color="#c0504d", alpha=0.7, label=f"Acacia (n={len(p)})")
    base(ax, label); ax.legend(loc="lower right", markerscale=5)
plt.tight_layout(); plt.savefig(FIG / "presences_vs_background.png", dpi=200); plt.show()

Footprint overlap on a 1 km grid: of the cells holding *Acacia*
presences, how many also hold background points (and vice versa)?

In [ ]:
def cells(df, size=1000):
    return set(zip((df.geometry.x // size).astype(int), (df.geometry.y // size).astype(int)))

rows = []
for label, p, b in groups:
    cp, cb = cells(p), cells(b)
    rows.append(dict(group=label, acacia=len(p), background=len(b),
                     ratio_bg_per_presence=round(len(b) / max(len(p), 1), 1),
                     acacia_cells_1km=len(cp), background_cells_1km=len(cb),
                     pct_acacia_cells_with_background=round(100 * len(cp & cb) / max(len(cp), 1), 1),
                     pct_background_cells_with_acacia=round(100 * len(cp & cb) / max(len(cb), 1), 1)))
counts = pd.DataFrame(rows)
counts.to_csv(TAB / "presence_background_counts.csv", index=False)
counts

## 5. Burned-area filter

Proposed rule (not a decision): remove an observation if it falls
inside an ICNF burned area whose fire started between the observation
date and the reference year 2023 — i.e. after the observation for
records up to 2023, and from 2023 up to the observation for later
records.

In [ ]:
fires_local = fires[fires.intersects(district.geometry.iloc[0])]
ref_start, ref_end = pd.Timestamp(f"{REF_YEAR}-01-01"), pd.Timestamp(f"{REF_YEAR}-12-31")

def burned_between(points):
    j = gpd.sjoin(points[["id", "date", "geometry"]], fires_local[["fire_date", "geometry"]], predicate="within")
    lo = j["date"].where(j["date"] < ref_start, ref_start)
    hi = j["date"].where(j["date"] > ref_end, ref_end)
    hit = j[(j["fire_date"] >= lo) & (j["fire_date"] <= hi)]
    return points["id"].isin(hit["id"])

acacia_w = acacia_w.assign(burned=burned_between(acacia_w))
bg_w = bg_w.assign(burned=burned_between(bg_w))
burn_table = pd.DataFrame([
    dict(set=name, n=len(df), removed=int(df.burned.sum()), pct_removed=round(100 * df.burned.mean(), 1))
    for name, df in [("Acacia", acacia_w), ("Background", bg_w)]])
burn_table.to_csv(TAB / "burned_filter_counts.csv", index=False)
print(f"Burned areas intersecting the district, 2021–2025: {len(fires_local)}")
burn_table

Context: observations inside *any* 2021–2025 burned area, by
observation year and fire year — including fires **after** 2023, which
the rule above does not remove (the 2023 image predates them).

In [ ]:
any_fire = gpd.sjoin(acacia_w[["id", "year", "observer", "geometry"]], fires_local[["Ano", "geometry"]], predicate="within")
print(f"Acacia in any 2021–2025 burned area: {any_fire['id'].nunique()} of {len(acacia_w)} "
      f"({100 * any_fire['id'].nunique() / len(acacia_w):.1f}%)")
print("Burned area in the district (ha):", fires_local.groupby("Ano")["AreaHaSIG"].sum().round(0).to_dict())
any_fire.groupby(["Ano", "year"]).size().unstack(fill_value=0)

In [ ]:
post_ref = acacia_w["id"].isin(any_fire.loc[any_fire["Ano"] > REF_YEAR, "id"])
fig, ax = plt.subplots(figsize=(9, 9))
fires_local.plot(ax=ax, column="Ano", categorical=True, legend=True, alpha=0.5, cmap="YlOrRd",
                 legend_kwds=dict(title="Fire year", loc="lower right"))
acacia_w[~acacia_w.burned & ~post_ref].plot(ax=ax, markersize=2, color="#4f81bd", alpha=0.6)
acacia_w[post_ref].plot(ax=ax, markersize=2, color="#7030a0", alpha=0.6)
acacia_w[acacia_w.burned].plot(ax=ax, markersize=30, color="black", marker="x")
base(ax, f"Acacia 2021–2025 and ICNF burned areas\n"
         f"black ×: removed by the rule ({int(acacia_w.burned.sum())}); "
         f"purple: burned after {REF_YEAR} ({int(post_ref.sum())}, kept)")
plt.tight_layout(); plt.savefig(FIG / "burned_areas_overlay.png", dpi=200); plt.show()

## Summary

Window 2021–2025, obscured coordinates excluded (43 *Acacia*, 8,242
background).

- **Class ratio:** 6,333 *Acacia* vs. 77,301 other-plant observations
  (12 background per presence).
- **Observer A also records other plants:** 33,065 background records
  (43% of the background), so a target-group background can share this
  observer's footprint. 97% of Observer A's *Acacia* 1 km cells also
  hold their background records; for other observers, 92%.
- **Footprints are not identical:** only 26% of background 1 km cells
  hold *Acacia* — the background covers much more of the district,
  especially from other observers.
- **Burned-area rule (fire between observation and 2023):** removes
  only 2 *Acacia* and 33 background points — 2021–2023 were light fire
  years in the district (≈2,100 ha burned).
- **But 2,184 *Acacia* observations (34.5%) lie in areas burned in
  2025** (≈70,600 ha intersecting the district), mostly Observer A's
  2021 records. Kept under the rule, since the 2023 image predates the
  fire — but any later imagery (e.g. TESSERA 2025) or a later
  reference year would see burned ground there.